# Part 1 — Dogs vs. Cats: three classifiers (Options A, B, C)

Single notebook for the final submission: **Option A** (HOG + SVM), **Option B** (multi-scale HOG + HSV histogram + PCA + LinearSVC), **Option C** (fine-tuned ResNet-50).

**Dataset layout** (same as course `src/config.py`): place Kaggle data under `data/part1/data/kaggle/` with `train/train/` and `test/test/`.

**Working directory:** run this notebook from the `final-submission/` folder so `REPO_ROOT = parent` resolves to the repository root.


In [ ]:
# Paths and dataset validation (no src.* imports)
from pathlib import Path

REPO_ROOT = Path().resolve().parent
DATA_ROOT = REPO_ROOT / "data"
PART1_KAGGLE_DIR = DATA_ROOT / "part1" / "data" / "kaggle"
PART1_TRAIN_DIR = PART1_KAGGLE_DIR / "train" / "train"
PART1_TEST_DIR = PART1_KAGGLE_DIR / "test" / "test"

IMG_SIZE_CLASSICAL = (128, 128)
IMG_SIZE_CNN = (224, 224)
CLASS_NAMES = ["cat", "dog"]
LABEL_MAP = {"cat": 0, "dog": 1}

OUTPUTS_DIR = REPO_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
MODELS_DIR = OUTPUTS_DIR / "models"
SUBMISSIONS_DIR = OUTPUTS_DIR / "submissions"
TESTS_DIR = OUTPUTS_DIR / "tests"

for d in (FIGURES_DIR, MODELS_DIR, SUBMISSIONS_DIR, TESTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert PART1_TRAIN_DIR.exists(), f"Missing: {PART1_TRAIN_DIR}"
assert PART1_TEST_DIR.exists(), f"Missing: {PART1_TEST_DIR}"
print("Part 1 paths OK:")
print(f"  train: {PART1_TRAIN_DIR}")
print(f"  test:  {PART1_TEST_DIR}")


## Shared imports and evaluation helpers

Metrics and plotting mirror the project `src/evaluation.py` and `src/visualization.py` (inlined here).


In [ ]:
import csv
import json
import os
import time
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from scipy.special import expit
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC, SVC
from skimage.feature import hog
from sklearn.utils import resample
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models, transforms
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Metrics (from src/evaluation.py)
# ---------------------------------------------------------------------------


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }


def compute_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    return confusion_matrix(y_true, y_pred)


def compare_models(results: dict) -> pd.DataFrame:
    df = pd.DataFrame(results).T
    df.index.name = "model"
    return df.sort_values("accuracy", ascending=False)


# ---------------------------------------------------------------------------
# Data loading (from src/utils.py)
# ---------------------------------------------------------------------------


def load_labeled_images(
    img_size: tuple,
    grayscale: bool,
    max_samples: int | None,
    return_ids: bool,
):
    images = []
    labels = []
    ids = []

    for class_name, label in LABEL_MAP.items():
        class_dir = PART1_TRAIN_DIR / f"{class_name}s"
        paths = sorted(class_dir.glob("*.jpg"))
        if max_samples is not None:
            paths = paths[:max_samples]

        for p in tqdm(paths, desc=f"Loading {class_name}s"):
            flag = cv2.IMREAD_GRAYSCALE if grayscale else cv2.IMREAD_COLOR
            img = cv2.imread(str(p), flag)
            if img is None:
                continue
            img = cv2.resize(img, (img_size[1], img_size[0]))
            if not grayscale:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
            labels.append(label)
            ids.append(p.stem)

    X = np.array(images, dtype=np.float32) / 255.0
    y = np.array(labels, dtype=np.int64)
    id_arr = np.array(ids, dtype=object)
    if return_ids:
        return X, y, id_arr
    return X, y


def load_test_images(img_size: tuple, grayscale: bool):
    paths = sorted(PART1_TEST_DIR.glob("*.jpg"), key=lambda p: int(p.stem))
    images = []
    ids = []

    for p in tqdm(paths, desc="Loading test images"):
        flag = cv2.IMREAD_GRAYSCALE if grayscale else cv2.IMREAD_COLOR
        img = cv2.imread(str(p), flag)
        if img is None:
            continue
        img = cv2.resize(img, (img_size[1], img_size[0]))
        if not grayscale:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        images.append(img)
        ids.append(int(p.stem))

    X = np.array(images, dtype=np.float32) / 255.0
    return X, ids


def split_data(X, y, test_size: float, random_state: int):
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)


def apply_pca(X_train, X_test, n_components: int):
    n_train = X_train.shape[0]
    n_test = X_test.shape[0]
    X_train_flat = X_train.reshape(n_train, -1)
    X_test_flat = X_test.reshape(n_test, -1)

    pca = PCA(n_components=n_components, random_state=42)
    X_train_pca = pca.fit_transform(X_train_flat)
    X_test_pca = pca.transform(X_test_flat)
    return X_train_pca, X_test_pca, pca


def standardize_features(X_train, X_test):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    return X_train_s, X_test_s, scaler


def generate_submission_csv(ids, predictions, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame({"id": ids, "label": predictions})
    df = df.sort_values("id").reset_index(drop=True)
    df.to_csv(output_path, index=False, lineterminator="\n")
    return output_path


# ---------------------------------------------------------------------------
# PyTorch dataloaders + GPU augmentation (from src/utils.py)
# ---------------------------------------------------------------------------


def get_pytorch_dataloaders(
    X_train,
    y_train,
    X_val,
    y_val,
    batch_size: int,
    img_size: tuple,
    normalize_train: bool,
    pin_memory: bool = False,
    num_workers: int = 0,
):
    _mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    _std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    def _to_tensor(X: np.ndarray, do_normalize: bool) -> torch.Tensor:
        t = torch.from_numpy(X).permute(0, 3, 1, 2).float().clone()
        if (t.shape[2], t.shape[3]) != img_size:
            t = torch.nn.functional.interpolate(
                t, size=img_size, mode="bilinear", align_corners=False
            )
        if do_normalize:
            t.sub_(_mean).div_(_std)
        return t

    train_t = _to_tensor(X_train, do_normalize=normalize_train)
    val_t = _to_tensor(X_val, do_normalize=True)

    train_ds = TensorDataset(train_t, torch.from_numpy(y_train).long())
    val_ds = TensorDataset(val_t, torch.from_numpy(y_val).long())

    loader_kw = {"batch_size": batch_size, "pin_memory": pin_memory, "num_workers": num_workers}
    if num_workers > 0:
        loader_kw["persistent_workers"] = True
        loader_kw["prefetch_factor"] = 2

    train_loader = DataLoader(train_ds, shuffle=True, **loader_kw)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_kw)
    return train_loader, val_loader


def build_gpu_augmentation(img_size: tuple):
    from torchvision import transforms as T

    return T.Compose(
        [
            T.RandomHorizontalFlip(),
            T.RandomRotation(15),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            T.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            T.RandomErasing(p=0.1),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


# ---------------------------------------------------------------------------
# Plots (from src/visualization.py)
# ---------------------------------------------------------------------------


def plot_confusion_matrix_fig(
    cm: np.ndarray,
    class_names,
    title: str,
    save_path=None,
):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    fig.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
    plt.show()


def plot_sample_predictions_fig(
    images: np.ndarray,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names,
    n: int,
    save_path=None,
):
    n = min(n, len(images))
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = axes.flatten()

    for i in range(n):
        ax = axes[i]
        img = images[i]
        if img.ndim == 2:
            ax.imshow(img, cmap="gray")
        else:
            ax.imshow(img)
        correct = y_true[i] == y_pred[i]
        color = "green" if correct else "red"
        ax.set_title(
            f"T:{class_names[y_true[i]]} P:{class_names[y_pred[i]]}",
            color=color,
            fontsize=9,
        )
        ax.axis("off")

    for j in range(n, len(axes)):
        axes[j].axis("off")

    fig.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
    plt.show()


def plot_model_comparison_fig(results: dict, save_path=None):
    df = pd.DataFrame(results).T
    ax = df.plot.bar(figsize=(10, 5), rot=25)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Internal validation — model configurations (Option B grid)")
    ax.legend(loc="lower right")
    fig = ax.get_figure()
    fig.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
    plt.show()


def plot_training_history_fig(history: dict, save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history["train_loss"]) + 1)

    ax1.plot(epochs, history["train_loss"], label="Train")
    ax1.plot(epochs, history["val_loss"], label="Val")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss")
    ax1.legend()

    ax2.plot(epochs, history["train_acc"], label="Train")
    ax2.plot(epochs, history["val_acc"], label="Val")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title("Accuracy")
    ax2.legend()

    fig.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
    plt.show()


print("Shared helpers loaded.")


## Option A — HOG + SVM (feature-based)

Resize to 64×64 RGB, **HOG** with `channel_axis=-1`, then **StandardScaler + SVC (RBF)** on a stratified 80/20 split.


In [ ]:
# Option A — HOG + SVM
VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

IMG_SIZE_A = (64, 64)
HOG_ORIENTATIONS_A = 9
HOG_PIXELS_PER_CELL_A = (8, 8)
HOG_CELLS_PER_BLOCK_A = (2, 2)
HOG_CHANNEL_AXIS_A = -1

SVM_C_A = 10.0
SVM_KERNEL_A = "rbf"
SVM_GAMMA_A = "scale"

TEST_SIZE_A = 0.20
RANDOM_STATE_A = 42


def load_image_a(path):
    img = Image.open(path).convert("RGB")
    img = img.resize(IMG_SIZE_A, Image.BILINEAR)
    return np.array(img, dtype=np.float32) / 255.0


def compute_hog_a(img):
    return hog(
        img,
        orientations=HOG_ORIENTATIONS_A,
        pixels_per_cell=HOG_PIXELS_PER_CELL_A,
        cells_per_block=HOG_CELLS_PER_BLOCK_A,
        channel_axis=HOG_CHANNEL_AXIS_A,
        feature_vector=True,
    )


train_root_a = PART1_TRAIN_DIR
print(f"Option A — loading from: {train_root_a}")

images_paths_a = []
labels_raw_a = []

subdirs_a = sorted([d for d in train_root_a.iterdir() if d.is_dir()])

if subdirs_a:
    for class_dir in subdirs_a:
        img_paths = [p for p in class_dir.rglob("*") if p.suffix.lower() in VALID_EXT]
        print(f"  {class_dir.name}: {len(img_paths)} images")
        for p in img_paths:
            images_paths_a.append(p)
            labels_raw_a.append(class_dir.name)
else:
    img_paths = [p for p in train_root_a.glob("*") if p.suffix.lower() in VALID_EXT]
    print(f"  flat folder: {len(img_paths)} images")
    for p in img_paths:
        images_paths_a.append(p)
        labels_raw_a.append(p.name.split(".")[0])

print(f"Total images: {len(images_paths_a)}")

X_list_a = []
for p in tqdm(images_paths_a, unit="img"):
    X_list_a.append(compute_hog_a(load_image_a(p)))

X_a = np.array(X_list_a, dtype=np.float32)

le_a = LabelEncoder()
y_a = le_a.fit_transform(labels_raw_a)
class_names_a = list(le_a.classes_)

print(f"HOG dim: {X_a.shape[1]}")
print(f"Labels: {dict(zip(le_a.classes_, le_a.transform(le_a.classes_)))}")

indices_a = np.arange(len(X_a))
X_train_a, X_test_a, y_train_a, y_test_a, train_idx_a, test_idx_a = train_test_split(
    X_a,
    y_a,
    indices_a,
    test_size=TEST_SIZE_A,
    random_state=RANDOM_STATE_A,
    stratify=y_a,
)

print(f"Train: {len(X_train_a)}, Test: {len(X_test_a)}")

model_a = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "svm",
            SVC(
                C=SVM_C_A,
                kernel=SVM_KERNEL_A,
                gamma=SVM_GAMMA_A,
                class_weight="balanced",
                decision_function_shape="ovr",
                random_state=RANDOM_STATE_A,
            ),
        ),
    ]
)

t0 = time.time()
model_a.fit(X_train_a, y_train_a)
print(f"Option A training time: {(time.time() - t0) / 60:.2f} min")

y_pred_a = model_a.predict(X_test_a)
metrics_a = compute_metrics(y_test_a, y_pred_a)
print("Option A metrics:", metrics_a)


def display_results_a(indices, title, num_samples=10):
    n = min(num_samples, len(indices))
    cols = 5
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(20, 5 * rows))
    for i in range(n):
        idx = indices[i]
        plt.subplot(rows, cols, i + 1)
        g_idx = test_idx_a[idx]
        img_path = images_paths_a[g_idx]
        img = mpimg.imread(img_path)
        plt.imshow(img)
        is_correct = y_pred_a[idx] == y_test_a[idx]
        color = "green" if is_correct else "red"
        plt.title(
            f"True: {class_names_a[y_test_a[idx]]}\nPred: {class_names_a[y_pred_a[idx]]}",
            fontsize=12,
            color=color,
            fontweight="bold",
        )
        plt.axis("off")
    plt.suptitle(title, fontsize=22, y=1.02)
    plt.tight_layout()
    plt.show()


correct_idx_a = np.where(y_pred_a == y_test_a)[0]
incorrect_idx_a = np.where(y_pred_a != y_test_a)[0]
acc_a = metrics_a["accuracy"]
acc_text = f" (Accuracy: {acc_a * 100:.2f}%)"
display_results_a(correct_idx_a, f"Option A — correct{acc_text}")
display_results_a(incorrect_idx_a, f"Option A — errors{acc_text}")

cm_a = compute_confusion_matrix(y_test_a, y_pred_a)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_a,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names_a,
    yticklabels=class_names_a,
    linewidths=0.5,
    ax=ax,
)
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
ax.set_title("Option A — HOG + SVM confusion matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## Option B — Appearance-based (multi-scale HOG + HSV + PCA + LinearSVC)

RGB **128×128**, two HOG scales **ppc=(8,8)** and **ppc=(16,16)**, **HSV** histogram (12 bins per channel), **StandardScaler**, **PCA**, **LinearSVC** (`dual=False`). Internal **80/20** stratified split; grid over PCA dimensions and `C` (same search as notebook 02).


In [ ]:
# Option B — feature extraction helpers
from sklearn.linear_model import LogisticRegression


def to_gray_uint8_b(img: np.ndarray) -> np.ndarray:
    if img.ndim == 3 and img.shape[2] == 3:
        gray = np.dot(img[..., :3], [0.2989, 0.5870, 0.1140])
    else:
        gray = img
    return (gray * 255).astype(np.uint8)


def extract_hsv_histogram_b(images: np.ndarray, bins: int = 12) -> np.ndarray:
    feats = []
    for img in tqdm(images, desc="HSV histogram"):
        rgb_u8 = (img * 255).astype(np.uint8)
        hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)
        hist_h, _ = np.histogram(hsv[..., 0], bins=bins, range=(0, 180), density=True)
        hist_s, _ = np.histogram(hsv[..., 1], bins=bins, range=(0, 256), density=True)
        hist_v, _ = np.histogram(hsv[..., 2], bins=bins, range=(0, 256), density=True)
        feats.append(np.concatenate([hist_h, hist_s, hist_v]))
    return np.array(feats, dtype=np.float32)


def extract_multiscale_hog_b(
    images: np.ndarray,
    orientations: int = 9,
    cells_per_block: tuple = (2, 2),
    scales_ppc: list = None,
) -> np.ndarray:
    if scales_ppc is None:
        scales_ppc = [(8, 8), (16, 16)]
    all_feats = []
    for img in tqdm(images, desc="Multi-scale HOG"):
        gray = to_gray_uint8_b(img)
        parts = []
        for ppc in scales_ppc:
            fd = hog(
                gray,
                orientations=orientations,
                pixels_per_cell=ppc,
                cells_per_block=cells_per_block,
                feature_vector=True,
            )
            parts.append(fd)
        all_feats.append(np.concatenate(parts))
    return np.array(all_feats, dtype=np.float32)


def extract_combined_features_b(
    images: np.ndarray,
    hog_orientations: int = 9,
    hog_cells_per_block: tuple = (2, 2),
    hog_scales_ppc: list = None,
    hsv_bins: int = 12,
) -> np.ndarray:
    if hog_scales_ppc is None:
        hog_scales_ppc = [(8, 8), (16, 16)]
    hog_feats = extract_multiscale_hog_b(
        images,
        orientations=hog_orientations,
        cells_per_block=hog_cells_per_block,
        scales_ppc=hog_scales_ppc,
    )
    hsv_feats = extract_hsv_histogram_b(images, bins=hsv_bins)
    return np.concatenate([hog_feats, hsv_feats], axis=1)


IMG_SIZE_OPT_B = IMG_SIZE_CLASSICAL
X_b, y_b, train_ids_b = load_labeled_images(
    img_size=IMG_SIZE_OPT_B,
    grayscale=False,
    max_samples=None,
    return_ids=True,
)
print(f"Loaded {X_b.shape}, labels {y_b.shape}, example ids {train_ids_b[:3]}")
print(f"Class counts: {np.bincount(y_b)}")

X_train_b, X_test_b, y_train_b, y_test_b = split_data(
    X_b, y_b, test_size=0.2, random_state=42
)
print(f"Train {X_train_b.shape}, test {X_test_b.shape}")


In [ ]:
# Option B — baseline raw-pixel PCA + LogReg (report comparison)
baseline_results_b = {}
for n_components in [100, 200]:
    X_train_pca, X_test_pca, _ = apply_pca(X_train_b, X_test_b, n_components=n_components)
    baseline = LogisticRegression(max_iter=1000, random_state=42)
    baseline.fit(X_train_pca, y_train_b)
    y_pred_b0 = baseline.predict(X_test_pca)
    metrics_b0 = compute_metrics(y_test_b, y_pred_b0)
    name_b = f"baseline PCA-{n_components}+LogReg"
    baseline_results_b[name_b] = metrics_b0
    print(f"{name_b} -> {metrics_b0}")


In [ ]:
# Option B — extract features, standardize, grid search
HOG_ORIENTATIONS_B = 9
HOG_CELLS_PER_BLOCK_B = (2, 2)
HOG_SCALES_PPC_B = [(8, 8), (16, 16)]
HSV_BINS_B = 12

print("Train features...")
X_train_feats_b = extract_combined_features_b(
    X_train_b,
    hog_orientations=HOG_ORIENTATIONS_B,
    hog_cells_per_block=HOG_CELLS_PER_BLOCK_B,
    hog_scales_ppc=HOG_SCALES_PPC_B,
    hsv_bins=HSV_BINS_B,
)
print("Test features...")
X_test_feats_b = extract_combined_features_b(
    X_test_b,
    hog_orientations=HOG_ORIENTATIONS_B,
    hog_cells_per_block=HOG_CELLS_PER_BLOCK_B,
    hog_scales_ppc=HOG_SCALES_PPC_B,
    hsv_bins=HSV_BINS_B,
)
print(f"Feature dim: {X_train_feats_b.shape[1]}")

X_train_s_b, X_test_s_b, feat_scaler_b = standardize_features(X_train_feats_b, X_test_feats_b)

pca_dims_b = [200, 300, 400, 500]
c_values_b = [0.01, 0.1, 1.0]

results_b = {}
for k, v in baseline_results_b.items():
    results_b[k] = v

best_accuracy_b = -1.0
best_n_components_b = None
best_C_b = None
best_pca_b = None
best_svm_b = None
best_y_pred_b = None

for n_components in pca_dims_b:
    pca = PCA(n_components=n_components, random_state=42)
    X_tr_pca = pca.fit_transform(X_train_s_b)
    X_te_pca = pca.transform(X_test_s_b)

    for C in c_values_b:
        svm = LinearSVC(C=C, max_iter=10000, dual=False, random_state=42)
        svm.fit(X_tr_pca, y_train_b)
        y_pred = svm.predict(X_te_pca)
        metrics = compute_metrics(y_test_b, y_pred)
        name = f"MultiHOG+HSV+PCA-{n_components}+LinearSVC-C={C}"
        results_b[name] = metrics
        print(f"{name} -> {metrics}")

        if metrics["accuracy"] > best_accuracy_b:
            best_accuracy_b = metrics["accuracy"]
            best_n_components_b = n_components
            best_C_b = C
            best_pca_b = pca
            best_svm_b = svm
            best_y_pred_b = y_pred

print()
print(
    f"Best Option B: PCA n={best_n_components_b}, C={best_C_b}, "
    f"accuracy={best_accuracy_b:.4f}"
)

metrics_b = compute_metrics(y_test_b, best_y_pred_b)
comparison_df_b = compare_models(results_b)
comparison_df_b


In [ ]:
# Option B — plots
plot_model_comparison_fig(results_b, save_path=FIGURES_DIR / "optionB_grid_metrics.png")

best_cm_b = compute_confusion_matrix(y_test_b, best_y_pred_b)
plot_confusion_matrix_fig(
    best_cm_b,
    CLASS_NAMES,
    title=(
        f"Option B best: MultiHOG+HSV+PCA-{best_n_components_b}"
        f"+LinearSVC (C={best_C_b})"
    ),
    save_path=FIGURES_DIR / "confusion_matrix_optionB.png",
)

plot_sample_predictions_fig(
    X_test_b,
    y_test_b,
    best_y_pred_b,
    CLASS_NAMES,
    n=16,
    save_path=FIGURES_DIR / "sample_predictions_optionB.png",
)

cumulative_ev_b = np.cumsum(best_pca_b.explained_variance_ratio_)
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(cumulative_ev_b) + 1), cumulative_ev_b)
plt.xlabel("Number of PCA components")
plt.ylabel("Cumulative explained variance")
plt.title(f"PCA (Option B), n={best_n_components_b}, C={best_C_b}")
plt.grid(True)
plt.savefig(FIGURES_DIR / "pca_variance_optionB.png", dpi=150)
plt.show()
print(f"Variance explained by {best_n_components_b} components: {cumulative_ev_b[-1]:.3f}")


In [ ]:
# Option B — retrain on all data, write Kaggle CSVs
print("Full-train features...")
X_all_feats_b = extract_combined_features_b(
    X_b,
    hog_orientations=HOG_ORIENTATIONS_B,
    hog_cells_per_block=HOG_CELLS_PER_BLOCK_B,
    hog_scales_ppc=HOG_SCALES_PPC_B,
    hsv_bins=HSV_BINS_B,
)

X_all_s_b, _, scaler_final_b = standardize_features(X_all_feats_b, X_all_feats_b)

pca_final_b = PCA(n_components=best_n_components_b, random_state=42)
X_all_pca_b = pca_final_b.fit_transform(X_all_s_b)

svm_final_b = LinearSVC(C=best_C_b, max_iter=10000, dual=False, random_state=42)
svm_final_b.fit(X_all_pca_b, y_b)
print("Option B final model fit on all labels.")

X_kaggle_b, kaggle_ids_b = load_test_images(img_size=IMG_SIZE_OPT_B, grayscale=False)
X_kaggle_feats_b = extract_combined_features_b(
    X_kaggle_b,
    hog_orientations=HOG_ORIENTATIONS_B,
    hog_cells_per_block=HOG_CELLS_PER_BLOCK_B,
    hog_scales_ppc=HOG_SCALES_PPC_B,
    hsv_bins=HSV_BINS_B,
)
X_kaggle_s_b = scaler_final_b.transform(X_kaggle_feats_b)
X_kaggle_pca_b = pca_final_b.transform(X_kaggle_s_b)
kaggle_pred_b = svm_final_b.predict(X_kaggle_pca_b)
kaggle_scores_b = svm_final_b.decision_function(X_kaggle_pca_b)
kaggle_pred_proba_b = expit(kaggle_scores_b)

submission_b = generate_submission_csv(
    kaggle_ids_b,
    kaggle_pred_b,
    SUBMISSIONS_DIR / "optionB_multihog_hsv_pca_linearsvc_submission.csv",
)
debug_b = generate_submission_csv(
    kaggle_ids_b,
    kaggle_pred_proba_b,
    TESTS_DIR / "optionB_multihog_hsv_pca_linearsvc_submission_proba.csv",
)
print("Submission:", submission_b)
print("Debug proba:", debug_b)


## Option C — Deep learning (ResNet-50)

**224×224** RGB, **ResNet-50** pretrained, frozen backbone then full fine-tune, **GPU augmentation**, progressive training sizes, **TTA** on Kaggle test.

**GPU (CUDA):** the next cell sets **cuDNN benchmark**, **TF32**, **mixed precision (AMP)**, **pin_memory** + **DataLoader workers**, and a larger **batch size** when `torch.cuda.is_available()`. Tweak `CUDA_DEVICE_INDEX`, `BATCH_SIZE_C`, `NUM_WORKERS_C`, or set `USE_AMP_C = False` if you need to debug.


In [ ]:
# Option C — load data and training setup
# Default first CUDA device; set CUDA_VISIBLE_DEVICES or CUDA_DEVICE_INDEX for multi-GPU.
CUDA_DEVICE_INDEX = 0

if torch.cuda.is_available():
    torch.cuda.set_device(CUDA_DEVICE_INDEX)
    device_c = torch.device("cuda", CUDA_DEVICE_INDEX)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
else:
    device_c = torch.device("cpu")

print(f"Device: {device_c}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(torch.cuda.current_device())}")
    print(f"  CUDA (this build): {torch.version.cuda}, torch {torch.__version__}")

# Throughput (CUDA): pinned host memory + worker processes prefetch batches; AMP uses Tensor Cores.
_cpu_n = os.cpu_count() or 4
NUM_WORKERS_C = min(8, max(0, _cpu_n - 1)) if torch.cuda.is_available() else 0
# If Jupyter on Windows raises multiprocessing errors from DataLoader, set NUM_WORKERS_C = 0.
PIN_MEMORY_C = torch.cuda.is_available()
USE_AMP_C = torch.cuda.is_available()
USE_TORCH_COMPILE_C = False

X_c, y_c = load_labeled_images(
    img_size=IMG_SIZE_CNN,
    grayscale=False,
    max_samples=None,
    return_ids=False,
)
print(f"Full dataset: {X_c.shape}, {(y_c == 0).sum()} cats, {(y_c == 1).sum()} dogs")

X_train_full_c, X_val_c, y_train_full_c, y_val_c = split_data(
    X_c, y_c, test_size=0.2, random_state=42
)

SAMPLE_SIZES_C = [500, 1000, 2000, 4000, len(X_train_full_c)]
BATCH_SIZE_C = 64 if torch.cuda.is_available() else 32
NUM_EPOCHS_HEAD_C = 8
NUM_EPOCHS_FULL_C = 12

print(f"Train/val: {X_train_full_c.shape} / {X_val_c.shape}")
print(f"Progressive sizes: {SAMPLE_SIZES_C}")
print(
    f"Loader: pin_memory={PIN_MEMORY_C}, num_workers={NUM_WORKERS_C}, "
    f"AMP={USE_AMP_C}, batch={BATCH_SIZE_C}"
)

gpu_augment_c = build_gpu_augmentation(IMG_SIZE_CNN)


def build_model_c(dev):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(num_features, 2),
    )
    model = model.to(dev)
    if USE_TORCH_COMPILE_C and hasattr(torch, "compile"):
        model = torch.compile(model)
    return model


def train_one_epoch_c(model, loader, criterion, optimizer, dev, augment=None, scaler=None, use_amp=False):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    cuda_amp = use_amp and dev.type == "cuda"

    for images, labels in tqdm(loader, desc="  Train", leave=False):
        images = images.to(dev, non_blocking=True)
        labels = labels.to(dev, non_blocking=True)
        if augment is not None:
            aug_list = []
            for img in images:
                aug_list.append(augment(img))
            images = torch.stack(aug_list)
        optimizer.zero_grad(set_to_none=True)
        if cuda_amp and scaler is not None:
            with torch.amp.autocast("cuda", enabled=True):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


def evaluate_c(model, loader, criterion, dev, use_amp=False):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    cuda_amp = use_amp and dev.type == "cuda"

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="  Val  ", leave=False):
            images = images.to(dev, non_blocking=True)
            labels = labels.to(dev, non_blocking=True)
            if cuda_amp:
                with torch.amp.autocast("cuda", enabled=True):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total


print("Option C helpers ready.")


In [ ]:
# Option C — progressive training
progressive_results_c = []
best_state_c = None

for n_samples in SAMPLE_SIZES_C:
    print(f"\n{'='*60}\nTraining with {n_samples} images (of {len(X_train_full_c)})\n{'='*60}")

    if n_samples < len(X_train_full_c):
        X_sub_c, y_sub_c = resample(
            X_train_full_c,
            y_train_full_c,
            n_samples=n_samples,
            stratify=y_train_full_c,
            random_state=42,
        )
    else:
        X_sub_c, y_sub_c = X_train_full_c, y_train_full_c

    train_loader_c, val_loader_c = get_pytorch_dataloaders(
        X_sub_c,
        y_sub_c,
        X_val_c,
        y_val_c,
        batch_size=BATCH_SIZE_C,
        img_size=IMG_SIZE_CNN,
        normalize_train=False,
        pin_memory=PIN_MEMORY_C,
        num_workers=NUM_WORKERS_C,
    )
    print(f"  batches train/val: {len(train_loader_c)} / {len(val_loader_c)}")

    model_c = build_model_c(device_c)
    scaler_c = torch.amp.GradScaler(
        "cuda", enabled=(USE_AMP_C and device_c.type == "cuda")
    )
    criterion_c = nn.CrossEntropyLoss(label_smoothing=0.05)
    head_params_c = [p for p in model_c.fc.parameters() if p.requires_grad]
    optimizer_c = optim.Adam(head_params_c, lr=1e-3, weight_decay=1e-4)
    scheduler_c = optim.lr_scheduler.CosineAnnealingLR(optimizer_c, T_max=NUM_EPOCHS_HEAD_C)

    history_c = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc_c = 0.0
    best_state_run = None

    print(f"  Phase 1: head only, {NUM_EPOCHS_HEAD_C} epochs")
    for epoch in range(1, NUM_EPOCHS_HEAD_C + 1):
        tl, ta = train_one_epoch_c(
            model_c,
            train_loader_c,
            criterion_c,
            optimizer_c,
            device_c,
            augment=gpu_augment_c,
            scaler=scaler_c,
            use_amp=USE_AMP_C,
        )
        vl, va = evaluate_c(
            model_c, val_loader_c, criterion_c, device_c, use_amp=USE_AMP_C
        )
        scheduler_c.step()
        history_c["train_loss"].append(tl)
        history_c["val_loss"].append(vl)
        history_c["train_acc"].append(ta)
        history_c["val_acc"].append(va)
        print(f"    Ep {epoch}/{NUM_EPOCHS_HEAD_C} TrL:{tl:.4f} TrA:{ta:.4f} VL:{vl:.4f} VA:{va:.4f}")
        if va > best_val_acc_c:
            best_val_acc_c = va
            best_state_run = {k: v.cpu().clone() for k, v in model_c.state_dict().items()}

    for param in model_c.parameters():
        param.requires_grad = True
    optimizer_ft_c = optim.Adam(model_c.parameters(), lr=1e-5, weight_decay=1e-4)
    scheduler_ft_c = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft_c, T_max=NUM_EPOCHS_FULL_C)

    print(f"  Phase 2: full fine-tune, {NUM_EPOCHS_FULL_C} epochs")
    for epoch in range(1, NUM_EPOCHS_FULL_C + 1):
        tl, ta = train_one_epoch_c(
            model_c,
            train_loader_c,
            criterion_c,
            optimizer_ft_c,
            device_c,
            augment=gpu_augment_c,
            scaler=scaler_c,
            use_amp=USE_AMP_C,
        )
        vl, va = evaluate_c(
            model_c, val_loader_c, criterion_c, device_c, use_amp=USE_AMP_C
        )
        scheduler_ft_c.step()
        history_c["train_loss"].append(tl)
        history_c["val_loss"].append(vl)
        history_c["train_acc"].append(ta)
        history_c["val_acc"].append(va)
        print(f"    Ep {epoch}/{NUM_EPOCHS_FULL_C} TrL:{tl:.4f} TrA:{ta:.4f} VL:{vl:.4f} VA:{va:.4f}")
        if va > best_val_acc_c:
            best_val_acc_c = va
            best_state_run = {k: v.cpu().clone() for k, v in model_c.state_dict().items()}

    progressive_results_c.append(
        {"n_train": n_samples, "best_val_acc": best_val_acc_c, "history": history_c}
    )
    print(f"  best val acc: {best_val_acc_c:.4f}")
    best_state_c = best_state_run

model_c.load_state_dict(best_state_c)
model_c = model_c.to(device_c)

print("\nProgressive summary:")
for r in progressive_results_c:
    print(f"  {r['n_train']:>6d} images -> val acc {r['best_val_acc']:.4f}")


In [ ]:
# Option C — curves, validation metrics, confusion, samples
final_history_c = progressive_results_c[-1]["history"]
plot_training_history_fig(
    final_history_c, save_path=FIGURES_DIR / "training_history_optionC.png"
)

sizes_c = [r["n_train"] for r in progressive_results_c]
accs_c = [r["best_val_acc"] for r in progressive_results_c]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes_c, accs_c, "o-", linewidth=2, markersize=8)
ax.set_xlabel("Number of training images")
ax.set_ylabel("Best validation accuracy")
ax.set_title("Data scaling curve — ResNet-50")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.3)
for s, a in zip(sizes_c, accs_c):
    ax.annotate(f"{a:.3f}", (s, a), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "data_scaling_curve_optionC.png", dpi=150)
plt.show()

model_c.eval()
all_preds_c = []
all_labels_c = []
with torch.no_grad():
    for images, labels in tqdm(val_loader_c, desc="Val inference"):
        images = images.to(device_c, non_blocking=True)
        if USE_AMP_C and device_c.type == "cuda":
            with torch.amp.autocast("cuda", enabled=True):
                outputs = model_c(images)
        else:
            outputs = model_c(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds_c.append(preds)
        all_labels_c.append(labels.numpy())

y_pred_c = np.concatenate(all_preds_c)
y_true_c = np.concatenate(all_labels_c)
metrics_c = compute_metrics(y_true_c, y_pred_c)
print("Option C metrics (validation split):", metrics_c)

cm_c = compute_confusion_matrix(y_true_c, y_pred_c)
plot_confusion_matrix_fig(
    cm_c,
    CLASS_NAMES,
    title="Option C — ResNet-50",
    save_path=FIGURES_DIR / "confusion_matrix_optionC.png",
)

plot_sample_predictions_fig(
    X_val_c,
    y_true_c,
    y_pred_c,
    CLASS_NAMES,
    n=16,
    save_path=FIGURES_DIR / "sample_predictions_optionC.png",
)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
torch.save(model_c.state_dict(), MODELS_DIR / "option_c_cnn.pth")
with open(MODELS_DIR / "metrics_optionC.json", "w") as f:
    json.dump(metrics_c, f, indent=2)


In [ ]:
# Option C — Kaggle test with TTA
X_test_c, test_ids_c = load_test_images(img_size=IMG_SIZE_CNN, grayscale=False)
print(f"Test: {X_test_c.shape}")

normalize_c = transforms.Normalize(
    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
)
X_test_base_c = torch.from_numpy(X_test_c).permute(0, 3, 1, 2).float()

tta_fns = [
    lambda t: t,
    lambda t: torch.flip(t, dims=[3]),
]

model_c.eval()
avg_probs_c = None
for tta_idx, tta_fn in enumerate(tta_fns):
    X_aug = tta_fn(X_test_base_c.clone())
    norm_list = []
    for img in X_aug:
        norm_list.append(normalize_c(img))
    X_norm = torch.stack(norm_list)
    test_ds_c = TensorDataset(X_norm)
    test_loader_c = DataLoader(
        test_ds_c,
        batch_size=BATCH_SIZE_C,
        shuffle=False,
        pin_memory=PIN_MEMORY_C,
        num_workers=NUM_WORKERS_C,
    )
    all_probs = []
    with torch.no_grad():
        for (images,) in tqdm(test_loader_c, desc=f"TTA {tta_idx + 1}/{len(tta_fns)}"):
            images = images.to(device_c, non_blocking=True)
            if USE_AMP_C and device_c.type == "cuda":
                with torch.amp.autocast("cuda", enabled=True):
                    outputs = model_c(images)
            else:
                outputs = model_c(images)
            probs = torch.softmax(outputs, dim=1).cpu()
            all_probs.append(probs)
    probs_tensor = torch.cat(all_probs, dim=0)
    if avg_probs_c is None:
        avg_probs_c = probs_tensor
    else:
        avg_probs_c = avg_probs_c + probs_tensor

avg_probs_c = avg_probs_c / len(tta_fns)
predictions_c = avg_probs_c.argmax(dim=1).numpy()

sub_c = generate_submission_csv(
    test_ids_c, predictions_c, OUTPUTS_DIR / "submission_optionC.csv"
)
print("Saved:", sub_c)


## Cross-model comparison and discussion

Same Part 1 task; **internal splits differ in preprocessing** (A: 64×64 HOG paths, B: 128×128 features, C: 224×224 CNN) but all use **stratified 80/20** with `random_state=42` where applicable. Use the table below for the report: rank by **accuracy** and comment on **bias** (precision/recall) and **failure modes** (qualitative grids above).


In [ ]:
# Compare Option A, B (best row), C on held-out metrics
three_way = {
    "A: HOG + SVM (20% holdout)": metrics_a,
    "B: MultiHOG+HSV+PCA+LinearSVC (20% holdout)": metrics_b,
    "C: ResNet-50 (20% val)": metrics_c,
}

comparison_three = compare_models(three_way)
comparison_three


In [ ]:
# Bar chart: primary three models
df3 = pd.DataFrame(three_way).T
ax = df3.plot.bar(figsize=(8, 4), rot=0)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Part 1 — Options A vs B vs C (internal metrics)")
ax.legend(loc="lower right")
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "comparison_ABC.png", dpi=150)
plt.show()


### Interpretation (for the report)

- **Option A** is fastest to train and interpretable (gradient histograms), but **lower ceiling** than learned appearance or deep features.
- **Option B** adds **color** and **multi-scale** structure plus **PCA** compression; usually **between A and C** on accuracy with moderate compute.
- **Option C** leverages **ImageNet** representations and data augmentation; **best validation metrics** here, at the cost of **GPU time** and **memory**.

**Which to ship to Kaggle:** all three CSVs are written under `outputs/`; pick the model with the best **public leaderboard** score if it differs from internal ranking (overfitting / distribution shift).
